# Dataset preprocessing

Run the cells from top to bottom. This notebook reads `data/raw data/Final_data.csv`, checks and cleans the records, filters language, preprocesses title and description, validates local thumbnails, then writes stratified train, validation, and test CSV files to `data/processed/`.

## 1. Setup and load data

In [13]:
from pathlib import Path
import html
import re
from collections import Counter

import pandas as pd
from langdetect import DetectorFactory, detect_langs
from sklearn.model_selection import train_test_split

DetectorFactory.seed = 42

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

RAW_FILE = ROOT / 'data' / 'raw data' / 'Final_data.csv'
INTERIM_DIR = ROOT / 'data' / 'interim'
PROCESSED_DIR = ROOT / 'data' / 'processed'
THUMBNAIL_DIR = ROOT / 'Thumbnail'

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_FILE, usecols=range(10))
print(f'Loaded {len(df):,} rows and {len(df.columns)} columns')
print(df.columns.tolist())
df.head()

Loaded 21,238 rows and 10 columns
['video_id', 'url', 'title', 'description', 'channel_name', 'views', 'likes', 'duration', 'final_label', 'local_thumbnail']


,video_id,url,title,description,channel_name,views,likes,duration,final_label,local_thumbnail
0,rN3OX3BVYmE,https://www.youtube.com/watch?v=rN3OX3BVYmE,Кто Первый Выберется из Непробиваемого Ящика ...,Покупай Мерч А4 «Ламба&Гелик» на официальном с...,A4,44305915,2945405,1954s,1,Thumbnail\rN3OX3BVYmE.jpg
1,i9nS9PcuFw4,https://www.youtube.com/watch?v=i9nS9PcuFw4,কাতার বিশ্বকাপে পশ্চিমা নারী দর্শকরা কি নিরাপদ...,ফেসবুকে আমাদের FOLLOW করুন: https://www.facebo...,আলোর পথ,29772,1764,295s,1,Thumbnail\i9nS9PcuFw4.jpg
2,6EtuOTq5TmM,https://www.youtube.com/watch?v=6EtuOTq5TmM,"Clean The Sports Car, Keep It!",We covered our cars completely in dirt and the...,Stokes Twins,7715320,65395,1474s,1,Thumbnail\6EtuOTq5TmM.jpg
3,Ds0Y4L8qkqw,https://www.youtube.com/watch?v=Ds0Y4L8qkqw,Press Start: Leveling Up Beyond the Screen | S...,"In this energetic and relatable talk, Shivaay ...",TEDx Talks,621,68,277s,0,Thumbnail\Ds0Y4L8qkqw.jpg
4,siDXCmJQgGc,https://www.youtube.com/watch?v=siDXCmJQgGc,TikToks That Are Actually Relatable,TikToks That Are Actually Relatable! Leave a L...,SSSniperWolf,1658896,23967,829s,1,Thumbnail\siDXCmJQgGc.jpg


In [14]:
VIDEO_ID_COL = 'video_id'
URL_COL = 'url'
TITLE_COL = 'title'
DESCRIPTION_COL = 'description'
CHANNEL_COL = 'channel_name'
THUMBNAIL_COL = 'local_thumbnail'
LABEL_COL = 'final_label'

required = [VIDEO_ID_COL, URL_COL, TITLE_COL, DESCRIPTION_COL, LABEL_COL, THUMBNAIL_COL]
missing_columns = [column for column in required if column not in df.columns]
if missing_columns:
    raise KeyError(f'Expected columns not found: {missing_columns}')

print(f'Total records: {len(df):,}')
for column in [VIDEO_ID_COL, URL_COL]:
    values = df[column].fillna('').astype(str).str.strip()
    duplicates = values.ne('') & values.duplicated(keep='first')
    print(f'Duplicate {column}: {int(duplicates.sum()):,}')

missing_titles = df[TITLE_COL].isna() | df[TITLE_COL].astype(str).str.strip().eq('')
print(f'Missing titles: {int(missing_titles.sum()):,}')

keep = ~missing_titles
for column in [VIDEO_ID_COL, URL_COL]:
    values = df[column].fillna('').astype(str).str.strip()
    keep &= values.eq('') | ~values.duplicated(keep='first')

df = df.loc[keep].copy().reset_index(drop=True)
print(f'Records after integrity checks: {len(df):,}')
df.to_csv(INTERIM_DIR / '01_integrity.csv', index=False)

Total records: 21,238
Duplicate video_id: 0
Duplicate url: 0
Missing titles: 0
Records after integrity checks: 21,238


In [15]:
def is_english_or_empty(value):
    if pd.isna(value) or not str(value).strip():
        return True
    text = str(value).strip()
    if len(re.findall(r'[A-Za-z]', text)) < 4:
        return True

    chunks = [part.strip() for part in re.split(r'[.!?\n]+', text) if len(re.findall(r'[A-Za-z]', part)) >= 12]
    samples = chunks if len(chunks) > 1 else [text]
    try:
        for sample in samples:
            languages = detect_langs(sample)
            if languages[0].lang != 'en' and languages[0].prob >= 0.75:
                return False
        return True
    except Exception:
        return True

def english_row(row):
    title_ok = is_english_or_empty(row[TITLE_COL])
    title_has_text = bool(str(row[TITLE_COL]).strip())
    description_ok = is_english_or_empty(row[DESCRIPTION_COL])
    return title_has_text and title_ok and description_ok

english_mask = df.apply(english_row, axis=1)
print(f'Removed non-English or mixed rows: {int((~english_mask).sum()):,}')
df = df.loc[english_mask].copy().reset_index(drop=True)
print(f'Records after language filter: {len(df):,}')
df.to_csv(INTERIM_DIR / '02_english.csv', index=False)

Removed non-English or mixed rows: 16,756
Records after language filter: 4,482


## 3. English language filter

Keep English titles and descriptions, including an English title with a missing description. Remove non-English or confidently mixed-language text. Short text that `langdetect` cannot identify is kept so titles such as “Wow!” are not discarded.

In [16]:
def is_english_or_empty(value):
    if pd.isna(value) or not str(value).strip():
        return True
    text = str(value).strip()
    if len(re.findall(r'[A-Za-z]', text)) < 4:
        return True
    try:
        return detect_langs(text)[0].lang == 'en'
    except Exception:
        return True

def english_row(row):
    title_ok = is_english_or_empty(row[TITLE_COL])
    title_has_text = bool(str(row[TITLE_COL]).strip())
    description_ok = is_english_or_empty(row[DESCRIPTION_COL])
    return title_has_text and title_ok and description_ok

english_mask = df.apply(english_row, axis=1)
print(f'Removed non-English or mixed rows: {int((~english_mask).sum()):,}')
df = df.loc[english_mask].copy().reset_index(drop=True)
print(f'Records after language filter: {len(df):,}')
df.to_csv(INTERIM_DIR / '02_english.csv', index=False)

Removed non-English or mixed rows: 257
Records after language filter: 4,225


## 4. Text processing

Only title and description are changed. Channel names and other columns stay as provided. URLs become the `URL` token; HTML tags are removed and HTML entities decoded. Repeated punctuation is capped at two characters.

In [17]:
URL_RE = re.compile(r'(?i)\b(?:https?://|www\.)\S+')
HTML_TAG_RE = re.compile(r'<[^>]+>')
REPEATED_PUNCTUATION_RE = re.compile(r'([!?.,])\1{2,}')

def clean_text(value):
    if pd.isna(value):
        return ''
    text = str(value)
    text = URL_RE.sub(' URL ', text)
    text = html.unescape(text)
    text = HTML_TAG_RE.sub(' ', text)
    text = REPEATED_PUNCTUATION_RE.sub(lambda match: match.group(1) * 2, text)
    return re.sub(r'\s+', ' ', text).strip()

df[TITLE_COL] = df[TITLE_COL].map(clean_text)
df[DESCRIPTION_COL] = df[DESCRIPTION_COL].map(clean_text)
df.to_csv(INTERIM_DIR / '03_text_processed.csv', index=False)
df[[TITLE_COL, DESCRIPTION_COL] + ([CHANNEL_COL] if CHANNEL_COL in df.columns else [])].head()

,title,description,channel_name
0,"Clean The Sports Car, Keep It!",We covered our cars completely in dirt and the...,Stokes Twins
1,Press Start: Leveling Up Beyond the Screen | S...,"In this energetic and relatable talk, Shivaay ...",TEDx Talks
2,Top 30 MCU Theories That Turned Out to Be TRUE,From Mjolnir's worthy wielder to the Quantum R...,WatchMojo.com
3,Cambodia says it shut nearly 200 scam centers ...,Cambodia has shut down nearly 200 scam centers...,Reuters
4,England hospitals face record flu patients | B...,England's Health Secretary Wes Streeting has s...,BBC News


## 5. Thumbnail checks

Thumbnail paths are resolved relative to the project root unless already absolute. The notebook reports duplicate references and missing files, then drops records with missing thumbnail files so the exported data has usable local thumbnail paths.

In [18]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")

def resolve_thumbnail(value):
    if pd.isna(value) or not str(value).strip():
        return None

    path = Path(str(value).strip())
    candidates = [path] if path.is_absolute() else [ROOT / path, THUMBNAIL_DIR / path.name]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    for candidate in candidates:
        for extension in IMAGE_EXTENSIONS:
            alternate = candidate.with_suffix(extension)
            if alternate.is_file():
                return alternate

    return None

thumbnail_paths = df[THUMBNAIL_COL].map(resolve_thumbnail)
valid_thumbnail = thumbnail_paths.notna()

normalized_paths = thumbnail_paths.map(
    lambda path: str(path.resolve()).casefold() if path else ""
)

duplicate_thumbnail = normalized_paths.ne("") & normalized_paths.duplicated(keep=False)
print(f"Duplicate thumbnail references: {int(duplicate_thumbnail.sum()):,} rows")
print(f"Missing or invalid thumbnails: {int((~valid_thumbnail).sum()):,} rows")

df = df.loc[valid_thumbnail].copy().reset_index(drop=True)
df[THUMBNAIL_COL] = thumbnail_paths.loc[valid_thumbnail].map(
    lambda path: path.relative_to(ROOT).as_posix()
)
df.to_csv(INTERIM_DIR / "04_thumbnail_checked.csv", index=False)
print(f"Records after thumbnail check: {len(df):,}")

Duplicate thumbnail references: 0 rows
Missing or invalid thumbnails: 0 rows
Records after thumbnail check: 4,225


## 6. Stratified 70/15/15 split

The label proportions are preserved in each split. The fixed seed makes the split repeatable.

In [19]:
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42

train_df, temporary_df = train_test_split(
    df, test_size=TEST_SIZE + VALIDATION_SIZE, random_state=RANDOM_STATE,
    stratify=df[LABEL_COL]
)
validation_fraction_of_temporary = VALIDATION_SIZE / (TEST_SIZE + VALIDATION_SIZE)
validation_df, test_df = train_test_split(
    temporary_df, test_size=1 - validation_fraction_of_temporary,
    random_state=RANDOM_STATE, stratify=temporary_df[LABEL_COL]
)

for name, split in [('train', train_df), ('validation', validation_df), ('test', test_df)]:
    print(f'{name}: {len(split):,} rows ({len(split) / len(df):.1%})')
    print(split[LABEL_COL].value_counts(normalize=True).sort_index().round(3).to_dict())
    split.to_csv(PROCESSED_DIR / f'{name}.csv', index=False)

print(f'Files saved in: {PROCESSED_DIR}')

train: 2,957 rows (70.0%)
{0: 0.789, 1: 0.211}
validation: 634 rows (15.0%)
{0: 0.789, 1: 0.211}
test: 634 rows (15.0%)
{0: 0.789, 1: 0.211}
Files saved in: d:\DC\multimodal-clickbait-detection\data\processed
